In [6]:
# 스크리너에서 수집한 종목코드, 종목명 데이터를 가져옵니다.
# 데이터 로드
import pandas as pd

df = pd.read_csv('nasdaq_screener_20260412.csv', na_filter=False)
df.head(3)

# 전처리
# df[df['Symbol'].str.contains(r'[^a-zA-X0-9-_/^ ]', regex=True)]
df['id'] = df['Symbol'].str.strip().replace(r'[/^]', '_', regex=True)
dict = df.to_dict(orient='records')


# stock_search() test

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()
MEILISEARCH_MASTER_KEY = os.environ['MEILISEARCH_MASTER_KEY']

In [11]:
from meilisearch import Client

client = Client("http://127.0.0.1:7700", MEILISEARCH_MASTER_KEY)

client.index('nasdaq').add_documents(dict, primary_key='id')

TaskInfo(task_uid=0, index_uid='nasdaq', status='enqueued', type='documentAdditionOrUpdate', enqueued_at=datetime.datetime(2026, 4, 12, 12, 54, 1, 966838))

In [8]:
from meilisearch import Client
client = Client("http://127.0.0.1:7700", MEILISEARCH_MASTER_KEY)

#종목 모드 검색
def stock_search(query: str):
    
    result = client.index('nasdaq').search(
        query,
        {'attributesToSearchOn': ['Symbol', 'Name']}
    )
    
    if (result['estimatedTotalHits']) :
        print (result['estimatedTotalHits'])
        return result['hits'][:10] # 10개 이내로 
    else :
        return []

In [10]:
result = stock_search("apple")
# result = stock_search("INVALID-TICKER")
result

9


[{'id': 'AAPL',
  'Symbol': 'AAPL',
  'Name': 'Apple Inc. Common Stock',
  'Last Sale': '$260.48',
  'Net Change': -0.01,
  '% Change': '-0.004%',
  'Market Cap': '3824143347200.00',
  'Country': 'United States',
  'IPO Year': '1980',
  'Volume': 31258020,
  'Sector': 'Technology',
  'Industry': 'Computer Manufacturing'},
 {'id': 'APLE',
  'Symbol': 'APLE',
  'Name': 'Apple Hospitality REIT Inc. Common Shares',
  'Last Sale': '$12.48',
  'Net Change': 0.08,
  '% Change': '0.645%',
  'Market Cap': '2945850249.00',
  'Country': 'United States',
  'IPO Year': '2015',
  'Volume': 2893247,
  'Sector': 'Real Estate',
  'Industry': 'Real Estate Investment Trusts'},
 {'id': 'AAOI',
  'Symbol': 'AAOI',
  'Name': 'Applied Optoelectronics Inc. Common Stock',
  'Last Sale': '$150.60',
  'Net Change': 17.3,
  '% Change': '12.978%',
  'Market Cap': '11324941840.00',
  'Country': 'United States',
  'IPO Year': '2013',
  'Volume': 21854437,
  'Sector': 'Technology',
  'Industry': 'Semiconductors'},
 {

In [ ]:
 # 종목 기본정보 스크래핑
import yfinance as yf
import pandas as pd   

symbol = "AAPL"
# symbol = "INVALID-TICKER"
ticker = yf.Ticker(symbol)

print("-" * 50)
print("info:", len(ticker.info), ticker.info)
# if not ticker.info:  # 또는 len(ticker.info) < 2
#     print("유효하지 않은 티커입니다.")
# else :
#     print(ticker.info['longName'])

# keys_to_keep = ['longName','industry','sector','marketCap','sharesOutstanding']

# sub_dict = {key: ticker.info[key] for key in keys_to_keep if key in ticker.info}

# sub_dict


--------------------------------------------------
info: 182 {'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store t

In [28]:
ticker.quarterly_income_stmt

,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31
Tax Effect Of Unusual Items,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Tax Rate For Calcs,1.750000e-01,1.627240e-01,1.640000e-01,1.550000e-01,1.470000e-01
Normalized EBITDA,5.406600e+10,3.555400e+10,3.103200e+10,3.225000e+10,4.591200e+10
Net Income From Continuing Operation Net Minority Interest,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10,3.633000e+10
Reconciled Depreciation,3.214000e+09,3.127000e+09,2.830000e+09,2.661000e+09,3.080000e+09
Reconciled Cost Of Revenue,7.452500e+10,5.412500e+10,5.031800e+10,5.049200e+10,6.602500e+10
EBITDA,5.406600e+10,3.555400e+10,3.103200e+10,3.225000e+10,4.591200e+10
EBIT,5.085200e+10,3.242700e+10,2.820200e+10,2.958900e+10,4.283200e+10
Normalized Income,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10,3.633000e+10
Net Income From Continuing And Discontinued Operation,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10,3.633000e+10


In [ ]:
ticker.quarterly_income_stmt.loc[
            ['Total Revenue','Gross Profit','Operating Income','Net Income']
        ].rename_axis('항목').rename(columns=lambda x: x.strftime("%Y-%m-%d"))

,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31
항목,,,,,
Total Revenue,1.437560e+11,1.024660e+11,9.403600e+10,9.535900e+10,1.243000e+11
Gross Profit,6.923100e+10,4.834100e+10,4.371800e+10,4.486700e+10,5.827500e+10
Operating Income,5.085200e+10,3.242700e+10,2.820200e+10,2.958900e+10,4.283200e+10
Net Income,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10,3.633000e+10


In [45]:
ticker.quarterly_income_stmt.loc[['Total Revenue','Gross Profit','Operating Income','Net Income']]

,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31
Total Revenue,1.437560e+11,1.024660e+11,9.403600e+10,9.535900e+10,1.243000e+11
Gross Profit,6.923100e+10,4.834100e+10,4.371800e+10,4.486700e+10,5.827500e+10
Operating Income,5.085200e+10,3.242700e+10,2.820200e+10,2.958900e+10,4.283200e+10
Net Income,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10,3.633000e+10


In [57]:
ticker.quarterly_income_stmt.loc[
            ['Total Revenue','Gross Profit','Operating Income','Net Income']
        ].rename_axis('항목').rename(columns=lambda x: x.strftime("%Y-%m-%d")).to_dict()

{'2025-12-31': {'Total Revenue': 143756000000.0,
  'Gross Profit': 69231000000.0,
  'Operating Income': 50852000000.0,
  'Net Income': 42097000000.0},
 '2025-09-30': {'Total Revenue': 102466000000.0,
  'Gross Profit': 48341000000.0,
  'Operating Income': 32427000000.0,
  'Net Income': 27466000000.0},
 '2025-06-30': {'Total Revenue': 94036000000.0,
  'Gross Profit': 43718000000.0,
  'Operating Income': 28202000000.0,
  'Net Income': 23434000000.0},
 '2025-03-31': {'Total Revenue': 95359000000.0,
  'Gross Profit': 44867000000.0,
  'Operating Income': 29589000000.0,
  'Net Income': 24780000000.0},
 '2024-12-31': {'Total Revenue': 124300000000.0,
  'Gross Profit': 58275000000.0,
  'Operating Income': 42832000000.0,
  'Net Income': 36330000000.0}}

In [76]:
a = ticker.quarterly_income_stmt.loc[
            ['Total Revenue','Gross Profit','Operating Income','Net Income']].rename_axis('항목').rename(columns=lambda x: x.strftime("%Y-%m-%d"))
a

,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31
항목,,,,,
Total Revenue,1.437560e+11,1.024660e+11,9.403600e+10,9.535900e+10,1.243000e+11
Gross Profit,6.923100e+10,4.834100e+10,4.371800e+10,4.486700e+10,5.827500e+10
Operating Income,5.085200e+10,3.242700e+10,2.820200e+10,2.958900e+10,4.283200e+10
Net Income,4.209700e+10,2.746600e+10,2.343400e+10,2.478000e+10,3.633000e+10


In [77]:
df = a.reset_index()
df.to_dict('records')


[{'항목': 'Total Revenue',
  '2025-12-31': 143756000000.0,
  '2025-09-30': 102466000000.0,
  '2025-06-30': 94036000000.0,
  '2025-03-31': 95359000000.0,
  '2024-12-31': 124300000000.0},
 {'항목': 'Gross Profit',
  '2025-12-31': 69231000000.0,
  '2025-09-30': 48341000000.0,
  '2025-06-30': 43718000000.0,
  '2025-03-31': 44867000000.0,
  '2024-12-31': 58275000000.0},
 {'항목': 'Operating Income',
  '2025-12-31': 50852000000.0,
  '2025-09-30': 32427000000.0,
  '2025-06-30': 28202000000.0,
  '2025-03-31': 29589000000.0,
  '2024-12-31': 42832000000.0},
 {'항목': 'Net Income',
  '2025-12-31': 42097000000.0,
  '2025-09-30': 27466000000.0,
  '2025-06-30': 23434000000.0,
  '2025-03-31': 24780000000.0,
  '2024-12-31': 36330000000.0}]

In [80]:

bal = ticker.quarterly_balance_sheet.loc[
            ['Total Assets','Total Liabilities Net Minority Interest','Stockholders Equity']
        ].rename_axis('항목').rename(columns=lambda x: x.strftime("%Y-%m-%d"))

bal = bal.iloc[:, :5]
bal


,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31
항목,,,,,
Total Assets,3.792970e+11,3.592410e+11,3.314950e+11,3.312330e+11,3.440850e+11
Total Liabilities Net Minority Interest,2.911070e+11,2.855080e+11,2.656650e+11,2.644370e+11,2.773270e+11
Stockholders Equity,8.819000e+10,7.373300e+10,6.583000e+10,6.679600e+10,6.675800e+10
